# Train a Dataset Builder zip

Trains yolov8n on this dataset and gives back `best.onnx`.

1. Set **Runtime → Change runtime type → GPU**.
2. Run the cells in order. The second asks for the zip: upload **your dataset zip**.

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files
up = files.upload()  # pick your dataset zip
zip_name = next(iter(up))
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{zip_name}" -d /content/ds
!cat /content/ds/data.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="/content/ds/data.yaml", imgsz=512, epochs=200, patience=20, batch=32, fliplr=0.0, mosaic=1.0, project="/content/runs", name="detect", exist_ok=True)

In [ ]:
# Fine-tune from the best weights at a lower learning rate. optimizer="auto" would ignore lr0, so it is named.
tuned = YOLO("/content/runs/detect/weights/best.pt")
tuned.train(data="/content/ds/data.yaml", imgsz=512, epochs=40, patience=10, batch=32, optimizer="AdamW", lr0=0.0002, lrf=0.1, cos_lr=True, warmup_epochs=1, fliplr=0.0, mosaic=1.0, project="/content/runs", name="detect-tune", exist_ok=True)

In [ ]:
# The fine-tune is kept only if it scores better on the valid split: it does not always help.
first = YOLO("/content/runs/detect/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
second = YOLO("/content/runs/detect-tune/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
run = "/content/runs/detect-tune/weights/" if second > first else "/content/runs/detect/weights/"
print("fine-tune", "kept" if second > first else "dropped", round(first, 4), "->", round(second, 4))
best = YOLO(run + "best.pt")
best.val(data="/content/ds/data.yaml", split="test", imgsz=512)
best.export(format="onnx", imgsz=512, opset=12, simplify=True)
files.download(run + "best.onnx")

The ONNX keeps no class names: they are `names` in `data.yaml`, in that order.

A run that stopped resumes with `YOLO("/content/runs/detect/weights/last.pt").train(resume=True)`, or `/content/runs/detect-tune/weights/last.pt` for the fine-tune.